# Train GOWA on Colab

**GOWA** (General OpenLex Worker Agent) is OpenLex's own decision model: mmBERT-small (MIT) with **one** Kev-style decision head that answers typed questions (yes/no, choice, score) with calibrated probabilities. This notebook runs the whole pipeline in `models/gowa/` — convert public data, train, calibrate, export to int8 ONNX, evaluate.

Rules:
- **Public data only.** Every example must come from a source with `include: yes` in `datasets.yaml`. Never upload client documents or anything from a matter.
- Use a GPU runtime (Runtime → Change runtime type → T4, L4 or A100). L4/A100 support bfloat16 (`--bf16`); on a T4 leave it off.
- Checkpoints go to Google Drive, so when the session ends you can re-run the training cell and it resumes.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RUN = '/content/drive/MyDrive/gowa/runs/gowa-small-v0'   # checkpoints and the final model
DATA = '/content/drive/MyDrive/gowa/data'                # converted examples

In [ ]:
!git clone --depth 1 https://github.com/N9394/openlex.git /content/openlex
%cd /content/openlex/models/gowa
!pip -q install -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Licence register
Everything below refuses sources that are not cleared.

In [ ]:
!python check_datasets.py

## 2. Convert public datasets
Pin dataset revisions with `--revision` for a reproducible run. LEXTREME subsets are only used after they were cleared and listed under `subsets` in `datasets.yaml`.

In [ ]:
import os; os.makedirs(DATA, exist_ok=True)
!python convert.py seed         --out {DATA}/seed.jsonl
!python convert.py cuad         --out {DATA}/cuad.jsonl
!python convert.py ledgar       --out {DATA}/ledgar.jsonl
!python convert.py unfair_tos   --out {DATA}/unfair_tos.jsonl
!python convert.py multi_eurlex --out {DATA}/multi_eurlex.jsonl --langs de en cs sk pl ro hu fr --limit 5000

## 3. (Optional) Labels from an open LLM on public law texts
Put public-text corpora (JSONL: `{"source": "<register id>", "lang": "de", "text": "..."}`) into `{DATA}/corpus/` — official statutes from the portals listed in `datasets.yaml`. Needs an L4/A100 for an 8B model. This is the main source for DE/CEE claim support, clause types, relevance and redaction examples.

In [ ]:
# !pip -q install vllm
# !python label_llm.py claim_support --corpus {DATA}/corpus/eurlex_de.jsonl --out {DATA}/claims_de.jsonl --limit 5000
# !python label_llm.py relevance     --corpus {DATA}/corpus/ris_at.jsonl   --out {DATA}/relevance_de.jsonl --limit 5000
# !python label_llm.py routing       --out {DATA}/routing.jsonl
# !python convert.py redaction --corpus {DATA}/corpus/eurlex_de.jsonl --out {DATA}/redaction_de.jsonl
# !python convert.py injection --corpus {DATA}/corpus/eurlex_de.jsonl --out {DATA}/injection_de.jsonl

## 4. Train (re-run this cell after a disconnect: it resumes from the last checkpoint)
mmBERT-small, one decision head, all tasks together. Ends with temperature fitting and per-task thresholds on the held-out set.

In [ ]:
!python train.py --data {DATA}/*.jsonl --out {RUN} \
    --base jhu-clsp/mmBERT-small --revision abc32620dd4f6ab06f5fbe905dc25f310618e09f \
    --max-len 512 --batch 16 --grad-accum 2 --epochs 2 --save-every 300 --bf16

In [ ]:
!cat {RUN}/final/calibration.json

## 5. Export to int8 ONNX (what the engine runs) and check it against PyTorch

In [ ]:
!python export_onnx.py --run {RUN} --out {RUN}/release && cat {RUN}/release/release.json

## 6. Evaluate per task and language
Use held-out or test-split files (`split: test`), never the training data.

In [ ]:
!python eval.py --onnx {RUN}/release --data {DATA}/*.jsonl --limit 500 --out {RUN}/eval
!cat {RUN}/eval/results.md

## 7. Try it in OpenLex
Copy `model.onnx`, `tokenizer.json`, `tokenizer_config.json` and `gowa-model.json` from `{RUN}/release` into `<OpenLex data folder>/models/gowa-small/` on your computer, then `openlex gowa status` and `openlex gowa on redaction` (or another task).

To publish: upload the release folder to Hugging Face, note the commit, and put `url`, `revision` and the `files` list from `release.json` into the manifest (`sidecar/src/gowa.js`) or a firm's `openlex.json` → `"decide": { "gowa": { "source": { … } } }`. Every file is checked against its SHA-256 on download.